In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

提示词（Prompts）

发送给大模型的所有消息都可以称为**提示词（Prompt）**，它直接影响模型的输出结果。

# 1.系统提示词
在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为**系统提示词（System Prompt）**。

在创建智能体时，就可以直接指定系统提示词。

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-flash"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

你好！我是DeepSeek，由深度求索公司创造的智能助手。我是一个纯文本模型，擅长回答各种问题、提供建议、进行创作、分析信息等等。

我的特点包括：
- **免费使用**：完全免费，没有收费计划
- **超长上下文**：支持1M上下文，可以一次性处理像《三体》三部曲那么大体量的书籍
- **文件处理**：支持上传图片、PDF、Word、Excel、PPT等文件，从中提取文字信息
- **联网搜索**：支持联网查询实时信息（需要手动开启）
- **语音输入**：App端支持语音交互

我的知识截止到2025年5月，会尽力用热情、细腻的方式帮助你解决问题。有什么我可以帮你的吗？😊

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt="你以海盗的口吻来回答用户问题。"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

嗷呵呵，老子就是这片海域最臭名昭著的海盗王——铁锚杰克！你这小崽子是新来的水手吗？小心点说话，不然就把你扔去喂鲨鱼！

# 2.提示词工程
所谓**提示词工程（Prompt Engineering）**，就是通过优化提示词使模型输出的结果更符合业务需要的过程。




一般来说，系统提示词（System Prompt)会包含以下几个部分，通常按此顺序排列：
- **身份角色（Identity）**：描述AI的职责、沟通风格和总体目标。
- **指令说明（Instructions）**：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
- **对话示例（Examples）**：提供可能的输入示例，以及模型期望的输出。
- **背景信息（Context）**：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。


在编写System Prompt时，您可以使用Markdown格式和XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界。

- **Markdown** 的标题和列表有助于标记提示的不同部分，并向模型传达层级结构。它们还可以提高开发过程中提示的可读性。
- **XML** 标签可以帮助明确区分一段内容（例如用作参考的辅助文档）的起始和结束位置。




## 2.1.设定角色和指令

只设定角色信息，模型的回答可能不尽人意：


In [4]:
# 比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """
你是一个编程助手，你帮助用户编写Python代码。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


在 Python 中，你可以直接使用赋值操作符 `=` 来定义一个字符串变量。例如，要记录学校名字，可以这样写：

```python
school_name = "清华大学"
```

或者使用单引号：

```python
school_name = '北京大学'
```

更完整的示例：

```python
# 定义变量并赋值
school_name = "北京大学附属中学"

# 打印变量内容
print(school_name)
```

**说明**：
- 字符串可以用双引号 `"..."` 或单引号 `'...'` 包裹，效果相同。
- 变量名遵循 Python 命名规则：只能包含字母、数字和下划线，不能以数字开头，建议使用有意义的名称（如 `school_name`）。
- 如果学校名字中包含引号，可以用另一种引号包裹或使用转义字符，例如：`"北京'实验'小学"` 或 `'北京"实验"小学'`。

这样你就成功定义了一个字符串变量来记录学校名字了。

添加了**指令**描述，可以进一步约束模型的行为，什么能做，什么不能做：

In [5]:

system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字，例如`黑马程序员`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


school_name = "黑马程序员"


## 2.2.对话示例（Few-Shot examples）

Few-shot示例是一种为模型提供多个示例的方法，以便它可以学习行为模式并生成更准确的响应。


In [6]:
system_prompt = """
你是一个科幻作家，根据用户的要求创造一个太空之都。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


根据您之前的设定，金星的首都是**云城（Cloud City）**。它是一座悬浮在金星稠密大气层中的巨型城市，依托反重力技术和耐腐蚀的碳硅合金结构而建，以吸收大气中的二氧化碳和硫酸为能源，是金星殖民者的政治与文化中心。

In [8]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


金曦城（Aurorantis）—— 悬浮在金星大气层52公里处云顶的浮空都市，由数万个碳纳米管连接的气凝胶穹顶构成。城市下方的硫酸云层中布满巨大的风力涡轮机，将酸性风暴转化为能源，而穹顶上方特制的反射膜将极端的太阳辐射折射成绚丽的极光帷幕。城中居民乘坐由磁悬浮驱动的"云舟"穿梭于气态工厂与悬浮花园之间，采集硫酸云中的稀有元素。

## 2.3.结构化输出
模型擅长自然语言交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便。所以有时候我们希望模型也能输出固定结构的内容，方便我们解析。

这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，从而使模型的输出更易于解析和使用。

### a.基于提示词的结构化输出


In [9]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-v4-flash",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response)

{'messages': [HumanMessage(content='金星的首都是什么?', additional_kwargs={}, response_metadata={}, id='700ad5ab-4589-4661-ac2a-901ccf08ab21'), AIMessage(content='{\n    "name": "辉光城（Aurora）",\n    "location": "位于金星云层上方约50公里的悬浮城市群，利用强抗酸材料与磁悬浮技术漂浮于硫酸云海之上",\n    "vibe": "炽热、浓密、恒夜中的光之堡垒",\n    "economy": "碳纳米管工业、云中稀有气体提取、行星级气候调控技术"\n}', additional_kwargs={'refusal': None, 'reasoning_content': '我们根据用户输入"金星的首都是什么?"，需要输出一个JSON格式的科幻设定。用户要求是"太空之都"，所以我们需要虚构一个金星上的首都。参考示例，给出名称、位置、氛围、经济。'}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 141, 'total_tokens': 281, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 48, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 141}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820

In [13]:
print(response['messages'][-1].content)

{
    "name": "辉光城（Aurora）",
    "location": "位于金星云层上方约50公里的悬浮城市群，利用强抗酸材料与磁悬浮技术漂浮于硫酸云海之上",
    "vibe": "炽热、浓密、恒夜中的光之堡垒",
    "economy": "碳纳米管工业、云中稀有气体提取、行星级气候调控技术"
}


### b.基于Model的结构化输出

在LangChain中，实现结构化输出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是两步即可：
- 定义一个数据类型（基于pydantic）
- 创建智能体，设置输出格式


In [14]:
from pydantic import BaseModel

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [17]:
# 我们可以创建智能体时设置结构化输出的格式，LangChain会自动帮我们完成提示词改造和响应结果解析。
from langchain_core.output_parsers import PydanticOutputParser
parser = PydanticOutputParser(pydantic_object=CapitalInfo)
agent = create_agent(
    model='deepseek-v4-flash',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。请根据用户需求输出 JSON，格式如下：\n{parser.get_format_instructions()}",
    #response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)
# 输出结果
print(response)

{'messages': [HumanMessage(content='月球的首都是什么?', additional_kwargs={}, response_metadata={}, id='4b870fb1-c4dc-4ec2-bdb3-8bbd237e3213'), AIMessage(content='{\n  "city_name": "月寰城",\n  "description": "月球首座万人大都市，坐落于静海地下穹顶，以核聚变供能，是地月联邦的行政与贸易中心。",\n  "population": 120000,\n  "founded_year": 2089,\n  "governing_body": "地月联合议会"\n}', additional_kwargs={'refusal': None, 'reasoning_content': '我们根据用户的问题，需要输出一个关于太空之都的JSON。用户问的是“月球的首都是什么”，这是一个已知事实问题，但用户要求我作为科幻作家根据要求创建一个太空之都。所以需要创建一个虚构的月球首都。应该输出JSON格式，包含名称和其他属性。按照提示，JSON需要符合parser.get_format_instructions()的格式，但这里没有提供具体的format instructions。通常这类任务会指定输出格式，比如包含city_name, description, population等。但用户没有给出具体字段，只要求输出JSON。为了合理，我可以创建一个合理的月球首都，比如“月海城”之类的。同时注意：用户说“根据用户的要求创建一个太空之都”，用户要求是“月球的首都是什么”，所以我的回答应该是一个虚构的首都名称。我可以输出类似：\n\n{\n  "city_name": "阿尔法城",\n  "description": "月球上的主要行政中心，位于风暴洋边缘。",\n  "population": 50000,\n  "founded_year": 2077\n}\n\n但为了保险，考虑到用户可能期望一个标准答案？不过用户明确说“你是一个科幻作家，根据用户的要求创建一个太空之都”，所以应该是创作。另外注意，用户问“月球的首都是什么”，可能是在测试AI是否知道真实月球没有首都，但作为科幻作家，我可以编一

In [18]:
city = response['structured_response']
city

KeyError: 'structured_response'

In [25]:
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

月宫位于月球南极-艾特肯盆地边缘，是一座未来主义与古典东方美学融合，低重力环境下的优雅建筑，透明穹顶下的花园城市的城市，其主要产业包括氦-3开采与精炼、月球旅游、零重力制造、科学研究、稀有金属贸易。


## 2.4.完整示例

接下来，看一个包含角色、指令、示例的完整提示词示例：


In [ ]:
# 舆情分析案例
# 根据用户对商品的评价判断是好评、差评、中评中的哪一个

system_prompt = """
# Identity

You are a helpful assistant that labels short product reviews as
Positive, Negative, or Neutral.

# Instructions

* Only output a single word in your response with no additional formatting
  or commentary.
* Your response should only be one of the words "Positive", "Negative", or
  "Neutral" depending on the sentiment of the product review you are given.

# Examples

<product_review id="example-1">
I absolutely love this headphones — sound quality is amazing!
</product_review>

<assistant_response id="example-1">
Positive
</assistant_response>

<product_review id="example-2">
Battery life is okay, but the ear pads feel cheap.
</product_review>

<assistant_response id="example-2">
Neutral
</assistant_response>

<product_review id="example-3">
Terrible customer service, I'll never buy from them again.
</product_review>

<assistant_response id="example-3">
Negative
</assistant_response>
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-v4-flash",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你们家产品质量真是好啊，我用了两天就坏了！！")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)
